# Kentucky Flash Flooding Events, Frequency, & Severity Data Analysis

## EDA: Flash Flooding Events in KY 2015 - 2025 Dataset

In [ ]:
# Import libraries and dependencies
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import seaborn as sns
import pandas as pd
import numpy as np
import sqlite3

In [ ]:
# Read dataset into dataframe
df = pd.read_csv('../data/raw/flash_floods_ky_2015_2025.csv')

In [ ]:
# Read first rows of dataset
df.head()

In [ ]:
# Read last rows of dataset
df.tail()

In [ ]:
# Check data type and missing values
df.info()

print(df.columns.tolist())

The EVENT_NARRATIVE column has 1749 non-null values. 

In [ ]:
# See the dimensions of the dataset
df.shape

In [ ]:
# Find categorical variables using .select_dtypes()
df.select_dtypes(include='object').columns

Columns such as BEGIN_DATE and END_DATE may need their data types changed since I plan to conduct numerical analysis with them. 

In [ ]:
# Explore value counts for various categorial variables
df.value_counts(subset=['flood_cause']),
df.value_counts(subset=['cz_name_str'])

print(df['flood_cause'].value_counts())
print(df['cz_name_str'].value_counts())

There is an outlier in the FLOOD_CAUSE column (Heavy Rain / Tropical System). 

In [ ]:
# Find row that has the Heavy Rain / Tropical System flood cause
heavy_rain_row = df.loc[df['flood_cause'] == 'Heavy Rain / Tropical System'].iloc[0]
print(heavy_rain_row)

In [ ]:
# Sort values using .sort_values()
df.sort_values(by='begin_date', ascending=True)

In [ ]:
# Rename columns using .rename()
df.rename(columns={'cz_name_str': 'county_name'}, inplace=True)

print(df.columns.tolist())

I should drop the MAGNITUDE, MAGNITUDE_TYPE, TOR_F_SCALE, TOR_LENGTH, and TOR_WIDTH columns since there aren't any values within those columns. Plus, these columns aren't relevant to the analysis that I'm conducting at this time.  

In [ ]:
# Drop columns using .drop()
df.drop(
    columns=[
        'magnitude',
        'magnitude_type',
        'tor_f_scale',
        'tor_length',
        'tor_width',
        'source_file',
        'absolute_rownumber'
    ],
    inplace=True
)

print(df.columns.tolist())

In [ ]:
# Fix data types using .astype()
df = df.astype({'begin_date': 'datetime64[ns]'})
df = df.astype({'end_date': 'datetime64[ns]'})
df = df.astype({'event_id': 'string'})
df = df.astype({'begin_time': 'string'})
df = df.astype({'end_time': 'string'})
df = df.astype({'episode_id': 'string'})
df = df.astype({'cz_fips': 'string'})
df = df.astype({'event_narrative': 'string'})
df = df.astype({'episode_narrative': 'string'})

print(df.dtypes)

I decided to change the data type of columns such as EVENT_ID, EVENT_NARRATIVE, and EPISODE_NARRATIVE to string since I will not be using these columns for numerical analysis at this time. 

In [ ]:
# Identify missing values using .isnull() and/or .notnull()
df.isnull().sum()

In [ ]:
# View rows with missing values
df[df.isnull().any(axis=1)] 

print(df[df['event_narrative'].isnull()])

EVENT_NARRATIVE values are missing from 07-19-2023. I'm not going to be using EVENT_NARRATIVE data at this time, so I will just fill those missing values in with "Unknown." 

In [ ]:
# Replace missing values in EVENT_NARRATIVE with "Unknown"
df["event_narrative"] = df["event_narrative"].fillna("Unknown")

print(df["event_narrative"].isna().sum())

In [ ]:
# Check for duplicate rows using .duplicated()
df.duplicated().sum()

In [ ]:
# Create YEAR column
df["year"] = df["begin_date"].dt.year

df = df.astype({'year': 'int64'})

print(df["year"].value_counts())

In [ ]:
# Check data to ensure that YEAR column was added correctly. 
df.info()

In [ ]:
# Save cleaned dataset to new csv file
df.to_csv('../data/processed/flash_floods_ky_2015_2025_cleaned.csv', index=False)

In [ ]:
# Use .groupby() to explore number of flash flooding events by county.
county_counts = (
    df.groupby("county_name")
    .size()
    .reset_index(name="total_events")
    .sort_values("total_events", ascending=False)
)

# View top 10 counties with most flash flooding events
print(county_counts.head(10))

In [ ]:
# Use .groupby() to explore total property damage by county
property_damage = (
    df.groupby("county_name")["damage_property_num"]
    .sum()
    .reset_index()
    .sort_values("damage_property_num", ascending=False)
)

# View top 10 counties with most property damage from flash flooding events
print(property_damage.head(10))

In [ ]:
# Use .groupby() to explore total deaths and injuries by county
human_impact = (
    df.groupby("county_name")[["deaths_direct", "injuries_direct"]]
    .sum()
    .reset_index()
    .sort_values("deaths_direct", ascending=False)
)

# View top 10 counties with the most direct deaths and injuries from flash flooding events
print(human_impact.head(10))

In [ ]:
# Use .groupby() to explore how many flash flooding events were reported from each source from 2015 - 2025
source_counts = (
    df.groupby("source")
      .size()
      .reset_index(name="total_events")
      .sort_values("total_events", ascending=False)
)

print(source_counts)

911 Call Centers have reported the most flash flooding events from 2015 - 2025.

In [ ]:
# Use .groupby() to explore the top 5 dates that had the most flash flooding events
date_counts = (
    df.groupby("begin_date")
      .size()
      .reset_index(name="total_events")
      .sort_values("total_events", ascending=False)
)

print(date_counts.head(5))

In [ ]:
# Use .groupby() to explore the counties and cities that experienced flash flooding events on the top 5 dates. Show the dates and the total number of events for each county on those dates.
top_dates = date_counts.head(5)["begin_date"]
county_city_date_counts = (
    df[df["begin_date"].isin(top_dates)]
    .groupby(["begin_date", "county_name", "begin_location", "begin_lat", "begin_lon"])
    .size()
    .reset_index(name="total_events")
    .sort_values(["begin_date", "total_events"], ascending=[True, False])
)

print(county_city_date_counts)

In [ ]:
# Sort values to determine which flash flooding events were the most expensive
highest_damage = df.sort_values("damage_property_num", ascending=False)
print(highest_damage[
    ["begin_date", "county_name", "damage_property_num"]
].head(10))

In [ ]:
# Sort Values to determine which flash flooding events had the most injuries
most_injuries = df.sort_values("injuries_direct", ascending=False)
print(most_injuries[
    ["begin_date", "county_name", "injuries_direct"]
].head(10))

In [ ]:
# Create a summary table for the following question: What has been the total statewide impact of flash flooding in Kentucky from 2015-2025?
summary = pd.DataFrame({
    "total events": [len(df)],
    "total property damage": [df["damage_property_num"].sum()],
    "total deaths": [df["deaths_direct"].sum()],
    "total injuries": [df["injuries_direct"].sum()]
})

print(summary)

In [ ]:
# Create a summary table for the following question: How has reporting sources changed over time?
source_year = pd.crosstab(df["year"], df["source"])

print(source_year)

This information can be used to create a heatmap of how reporting has changed for each source over time. 

In [ ]:
# Create a heatmap that shows correlations between numerical variables
events_df = pd.read_csv('../data/processed/flash_floods_ky_2015_2025_cleaned.csv')

cols = [
    "year",
    "deaths_direct",
    "injuries_direct",
    "damage_property_num",
    "damage_crops_num"
]
corr_matrix = events_df[cols].corr(method="pearson")

plt.figure(figsize=(8, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",  
    vmin=-1,
    vmax=1,
    linewidths=1,
    linecolor="white",
    cbar_kws={"label": "Pearson Correlation Coefficient (r)"},
)

plt.suptitle(
    "No Strong Correlations Found Between \n Year and Flash Flooding Impact Variables",
    fontsize=12,
    fontweight="bold",
    y=0.96,
)

plt.xticks(
    rotation=45, 
    ha="right"
)

plt.tight_layout()
plt.show()

1. What question or idea does this chart help answer?
This chart shows whether there are linear relationships between year and key flash flood impact variables, as well as relationships among the impact variables themselves. It helps identify whether any variables tend to increase, decrease, or move together over time.
2. Why is this chart type appropriate?
A correlation heatmap is appropriate because it allows multiple relationships to be visualized in a single view. It summarizes how several numeric variables relate to one another without needing many separate plots.
3. What design choices did you make (color, labels, order, scale)?
I used a diverging “coolwarm” color palette with a fixed range from -1 to 1 to clearly show direction and strength of correlations. I rotated labels for readability. I kept axis names consistent to make comparisons easier.
4. How did you ensure accuracy and avoid misleading design?
I ensured that the heatmap used Pearson correlation values directly from the dataset without transformation, ensuring mathematical accuracy. The full -1 to 1 scale and consistent labeling aids in preventing exaggeration or misinterpretation of relationships.

In [ ]:
# Create a heatmap that shows correlations between years and sources
events_df = pd.read_csv('../data/processed/flash_floods_ky_2015_2025_cleaned.csv')

source_year = pd.crosstab(events_df["year"], events_df["source"])

plt.figure(figsize=(14, 10))

ax = sns.heatmap(
    source_year, 
    annot=True, 
    cmap="YlGnBu", 
    fmt="d"
)

cbar = ax.collections[0].colorbar
cbar.set_label(
    "Number of Flash Flood Events", 
    labelpad=10)

plt.title(
    "911 Call Centers Have Reported the Most Flash Flooding Events from 2015-2025", 
    fontsize=14, 
    fontweight="bold", 
    pad=15,
    ha="center"
)

plt.xlabel(
    "Source", 
    fontsize=12, 
    labelpad=10
)

plt.ylabel(
    "Year", 
    fontsize=12, 
    labelpad=10
)

ax.set_xticklabels(
    source_year.columns, 
    rotation=65, 
    ha="right"
)

ax.set_yticklabels(
    source_year.index, 
    rotation=0, 
    ha="right"
)

offset = mtransforms.ScaledTranslation(8 / 78, 0, plt.gcf().dpi_scale_trans)

for label in ax.get_xticklabels():
    label.set_transform(label.get_transform() + offset)

plt.tight_layout()
plt.show()

1. What question or idea does this chart help answer?
This chart shows how the number of flash flood events varies by reporting source across each year. It helps identify which sources, such as 911 call centers, report the most events over time.
2. Why is this chart type appropriate?
A heatmap is appropriate because it allows comparison of counts across two categorical variables (year and source) in a compact visual format. It makes patterns in reporting volume easy to spot across both dimensions.
3. What design choices did you make (color, labels, order, scale)?
I chose to use the “YlGnBu” color palette to represent increasing event counts, with annotations added to show exact values in each cell. I made adjustments to axis labels, rotations, and spacing adjustments to improve readability of densely packed categories.
4. How did you ensure accuracy and avoid misleading design?
I ensured that the chart used raw counts from a crosstab without normalization so that values directly reflected event totals. I used Consistent labeling, clear axis definitions, and annotated values to help prevent misinterpretation of relative differences.

## Primary Question: Has the Frequency of Flash Flooding Events Increased in Kentucky from 2015–2025?

In [ ]:
# Create a line graph that shows total flash flooding events per year from 2015-2025
events_df = pd.read_csv('../data/processed/flash_floods_ky_2015_2025_cleaned.csv')

events_per_year = events_df["year"].value_counts().sort_index()

plt.figure(figsize=(12, 6))

ax = plt.gca()
ax.plot(
    events_per_year.index, events_per_year.values, 
    marker="o", linestyle="--", 
    linewidth=1, 
    color="#6272dc"
)

ax.set_title(
    "Has the Frequency of Flash Flooding Events\nIncreased in Kentucky from 2015 – 2025?", 
    fontsize=16, 
    fontweight="bold", 
    pad=18
)

ax.set_xlabel(
    "Year", 
    fontsize=12, 
    labelpad=10
)

ax.set_ylabel(
    "Total Events", 
    fontsize=12, 
    labelpad=10
)

ax.set_xticks(events_per_year.index)
ax.set_xticklabels(
    events_per_year.index, 
    rotation=45, 
    ha="right", 
    rotation_mode="anchor", 
    fontsize=11
)

ax.set_ylim(0, events_per_year.max() + 5)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.show()

1. What question or idea does this chart help answer?
This chart examines whether the frequency of flash flooding events in Kentucky has increased over time from 2015 to 2025. It highlights year-to-year trends in total event counts to show overall change patterns.
2. Why is this chart type appropriate?
A line chart is appropriate because it effectively shows trends over time and makes it easy to see increases or decreases across consecutive years. It is well-suited for time-series data like annual event counts.
3. What design choices did you make (color, labels, order, scale)?
I decided to use single blue line with markers and a dashed style to clearly show both trend and individual data points, while axis labels and rotated year ticks were changed to improve readability. I adjusted the y-axis to start at zero to avoid exaggerating changes.
4. How did you ensure accuracy and avoid misleading design?
I used sorted yearly counts to maintain correct chronological order and included a consistent y-axis scale starting at zero. I also removed unnecessary spines and used clear labeling to prevent visual distortion or misinterpretation of trends.

It appears that there hasn't been a significant increase in flash flooding frequency in Kentucky from 2015 - 2025.

In [ ]:
# Create a bar chart that shows the top 10 counties that have experienced the most flash flooding events from 2015 - 2025
events_df = pd.read_csv('../data/processed/flash_floods_ky_2015_2025_cleaned.csv')

county_counts = (
    events_df.groupby("county_name")
    .size()
    .reset_index(name="total_events")
    .sort_values("total_events", ascending=False)
    .head(10)
)

plt.figure(figsize=(12, 8))

ax = sns.barplot(
    data=county_counts, 
    x="county_name", 
    y="total_events", 
    color="steelblue"
)

plt.title(
    "Which County Has Experienced the Most Flash Flooding Events from 2015-2025?", 
    fontsize=16, 
    fontweight="bold", 
    pad=10
)

plt.xlabel(
    "County", 
    fontsize=12, 
    labelpad=8
)

plt.ylabel(
    "Total Events", 
    fontsize=12, 
    labelpad=8
)

plt.xticks(
    rotation=35, 
    ha="right", 
    rotation_mode="anchor", 
    fontsize=11
)

plt.yticks(fontsize=11)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

1. What question or idea does this chart help answer?
This chart identifies which top 10 counties in Kentucky experienced the highest number of flash flooding events from 2015 to 2025. It helps compare event frequency across these top 10 counties.
2. Why is this chart type appropriate?
A bar chart is appropriate because it clearly compares discrete categories (counties) based on their total event counts. It makes it easy to rank and visually distinguish differences between counties.
3. What design choices did you make (color, labels, order, scale)?
I decided to use a single “steelblue” color for the bars to keep focus on comparison rather than aesthetics. I sorted counties in descending order to highlight the highest-impact locations. I adjusted axis labels, rotation, and font sizing to improve readability.
4. How did you ensure accuracy and avoid misleading design?
I grouped and sorted directly from raw event counts without transformation, ensuring accurate totals for each county. I limited the plot to the top 10 counties in order to prevent clutter while still accurately representing the highest values.

## EDA: Flash Floods KY Weather Conditions Dataset

In [ ]:
# Read dataset into dataframe
df = pd.read_csv('../data/raw/flash_floods_ky_weather_conditions.csv')

In [ ]:
# Read first rows of dataset
df.head()

In [ ]:
# Read last rows of dataset
df.tail()

In [ ]:
# Check data type and missing values
df.info()

print(df.columns.tolist())

In [ ]:
# See the dimensions of the dataset
df.shape

In [ ]:
# Find categorical variables using .select_dtypes()
df.select_dtypes(include='object').columns

In [ ]:
# Drop columns using .drop()
df.drop(columns=[
    'event_type',
    'state_abbr', 
    'episode_id', 
    'cz_type', 
    'cz_fips', 
    'cz_timezone', 
    'wfo', 
    'source', 
    'flood_cause', 
    'begin_azimuth', 
    'end_azimuth', 
    'event_narrative', 
    'episode_narrative'
], inplace=True)

print(df.columns.tolist())

In [ ]:
# Check to see if columns dropped successfully
df.info()

In [ ]:
# Save Cleaned Dataset
df.to_csv('../data/processed/flash_floods_ky_weather_conditions_data.csv', index=False)

In [ ]:
# Create Flash Floods KY Events Info CSV file
df = pd.read_csv('../data/processed/flash_floods_ky_weather_conditions_data.csv')

df.drop(columns=[
    "maxtemp_f", 
    "mintemp_f", 
    "avgtemp_f", 
    "maxwind_mph", 
    "totalprecip_in", 
    "avgvis_miles", 
    "avghumidity", 
    "condition_text", 
    "condition_code", 
    "uv", 
    "daily_will_it_rain", 
    "daily_chance_of_rain"
], inplace=True)

df.to_csv('../data/processed/flash_floods_ky_event_info.csv', index=False)

df.info()

# Flash Floods Ky Event Info Dataset Cleaning

In [ ]:
# Read dataset into dataframe
df = pd.read_csv('../data/processed/flash_floods_ky_event_info.csv')
df.info()

In [ ]:
# Convert event_id to string
df['event_id'] = df['event_id'].astype(str)

In [ ]:
# Convert date strings to datetime objects
b_date = pd.to_datetime(df['begin_date'], format='mixed')
e_date = pd.to_datetime(df['end_date'], format='mixed')

In [ ]:
# Standardize time ints into 4-digit strings
b_time_str = df['begin_time'].astype(str).str.zfill(4)
e_time_str = df['end_time'].astype(str).str.zfill(4)

In [ ]:
# Create unified temporary datetime series
start_dt = pd.to_datetime(b_date.dt.strftime('%Y-%m-%d') + ' ' + b_time_str, format='%Y-%m-%d %H%M')
end_dt = pd.to_datetime(e_date.dt.strftime('%Y-%m-%d') + ' ' + e_time_str, format='%Y-%m-%d %H%M')

In [ ]:
# Calculate precise durations
df['duration_minutes'] = (end_dt - start_dt).dt.total_seconds() / 60.0
df['duration_hours'] = df['duration_minutes'] / 60.0

In [ ]:
# Round durations to 2 decimal places
df['duration_hours'] = df['duration_hours'].round(2)

In [ ]:
# Save updated DataFrame to CSV file
df.to_csv('../data/processed/flash_floods_ky_event_info.csv', index=False)

# Flash Floods Ky Impact Dataset Creation & Cleaning

In [ ]:
# Read dataset into dataframe
df = pd.read_csv('../data/processed/flash_floods_ky_event_info.csv')
df.info()

In [ ]:
# Convert event_id to string
df['event_id'] = df['event_id'].astype(str)

In [ ]:
# Drop columns
df.drop(columns=[
    "begin_location", 
    "begin_date", 
    "begin_time",
    "begin_range", 
    "end_range", 
    "end_location", 
    "end_date", 
    "end_time", 
    "begin_lat", 
    "begin_lon", 
    "end_lat", 
    "end_lon", 
    "year", 
    "duration_minutes",
    "duration_hours"
], inplace=True)

df.info()

In [ ]:
# Save to new CSV file
df.to_csv('../data/processed/flash_floods_ky_impact_data.csv', index=False)

# Build SQL Relational Tables

In [ ]:
df_event_info = pd.read_csv("../data/processed/flash_floods_ky_event_info.csv")
df_impact_data = pd.read_csv("../data/processed/flash_floods_ky_impact_data.csv")
df_weather_conditions = pd.read_csv("../data/processed/flash_floods_ky_weather_conditions_data.csv")
df_moon_data = pd.read_csv("../data/processed/flash_floods_ky_moon_data.csv")
df_oni_data = pd.read_csv("../data/processed/flash_floods_ky_oni_data.csv")
df_nlcd_data = pd.read_csv("../data/processed/flash_floods_ky_nlcd_data.csv")

conn = sqlite3.connect("../sql/ky_flash_floods_db.db")

df_event_info = df_event_info[[
    'event_id', 'county_name', 'begin_date', 'end_date', 'begin_time', 'end_time', 'begin_location', 'end_location', 
    'begin_range', 'end_range', 'begin_lat', 'begin_lon', 'end_lat', 'end_lon', 'year', 'duration_minutes', 'duration_hours'
]]

df_impact_data = df_impact_data[[
    'event_id', 'county_name', 'deaths_direct', 'injuries_direct', 'damage_property_num', 'damage_crops_num', 
    'injuries_indirect', 'deaths_indirect'
]]

df_weather_conditions = df_weather_conditions[[
    'event_id', 'maxtemp_f', 'mintemp_f', 'avgtemp_f', 'maxwind_mph', 
    'totalprecip_in', 'avgvis_miles', 'avghumidity', 'condition_text', 
    'condition_code', 'uv', 'daily_will_it_rain', 'daily_chance_of_rain'
]]

df_moon_data = df_moon_data[[
    'event_id', 'year', 'moon_phase_name', 'moon_illumination_pct'
]]

df_oni_data = df_oni_data[[
    'event_id', 'oni_season', 'year', 'oni_anomaly', 'enso_phase'
]]

df_nlcd_data = df_nlcd_data[[
    'event_id', 'nlcd_code', 'nlcd_class', 'elevation_m', 'impervious_surface_pct'
]]

df_event_info.to_sql("event_info", conn, index=False, if_exists='replace')
df_impact_data.to_sql("impact_data",conn, index=False, if_exists='replace')
df_weather_conditions.to_sql("weather_conditions", conn, index=False, if_exists='replace')
df_moon_data.to_sql("moon_data", conn, index=False, if_exists='replace')
df_oni_data.to_sql("oni_data", conn, index=False, if_exists='replace')
df_nlcd_data.to_sql("nlcd_data", conn, index=False, if_exists='replace')

conn.commit()

# SQL Queries and Advanced Data Visualizations

In [ ]:
# Create a SQL query to analyze the relationship between flash flooding events, precipitation, and impervious surfaces by NLCD class.
query1 = """ SELECT 
    n.nlcd_class, 
    COUNT(e.event_id) AS total_events, 
    ROUND(AVG(w.totalprecip_in), 2) AS avg_precipitation_inches, 
    ROUND(AVG(n.impervious_surface_pct), 2) AS avg_impervious_surface_pct
FROM event_info e
JOIN weather_conditions w ON e.event_id = w.event_id
JOIN nlcd_data n ON e.event_id = n.event_id
GROUP BY n.nlcd_class
ORDER BY total_events DESC;
"""
result1 = pd.read_sql(query1, conn)
result1

In [ ]:
# Create a SQL query to analyze the relationship between flash flooding events, precipitation, wind gusts, and UV index during the summer months (June, July, August) by county.
query2 = """ SELECT 
e.county_name, 
    COUNT(e.event_id) AS summer_flood_count, 
    ROUND(AVG(w.totalprecip_in), 2) AS avg_summer_rainfall, 
    MAX(w.maxwind_mph) AS peak_wind_gust_mph, 
    ROUND(AVG(w.uv), 1) AS avg_uv_index
FROM event_info e
JOIN weather_conditions w ON e.event_id = w.event_id
WHERE 
    strftime('%m', e.begin_date) IN ('06', '07', '08')
    OR e.begin_date LIKE '%-06-%' 
    OR e.begin_date LIKE '%-07-%' 
    OR e.begin_date LIKE '%-08-%'
GROUP BY e.county_name
HAVING COUNT(e.event_id) >= 2
ORDER BY avg_uv_index DESC;
"""
result2 = pd.read_sql(query2, conn)
result2

In [ ]:
# Create a SQL query to analyze the relationship between flash flooding events and moon illumination percentage by creating bins of 10% increments (0-10%, 10-20%, ..., 90-100%) and counting the number of events that occurred within each bin.
query3 = """ SELECT
CASE 
    WHEN m.moon_illumination_pct >= 0 AND m.moon_illumination_pct <= 10 THEN '0-10%'
    WHEN m.moon_illumination_pct > 10 AND m.moon_illumination_pct <= 20 THEN '10-20%'
    WHEN m.moon_illumination_pct > 20 AND m.moon_illumination_pct <= 30 THEN '20-30%'
    WHEN m.moon_illumination_pct > 30 AND m.moon_illumination_pct <= 40 THEN '30-40%'
    WHEN m.moon_illumination_pct > 40 AND m.moon_illumination_pct <= 50 THEN '40-50%'
    WHEN m.moon_illumination_pct > 50 AND m.moon_illumination_pct <= 60 THEN '50-60%'
    WHEN m.moon_illumination_pct > 60 AND m.moon_illumination_pct <= 70 THEN '60-70%'
    WHEN m.moon_illumination_pct > 70 AND m.moon_illumination_pct <= 80 THEN '70-80%'
    WHEN m.moon_illumination_pct > 80 AND m.moon_illumination_pct <= 90 THEN '80-90%'
    WHEN m.moon_illumination_pct > 90 AND m.moon_illumination_pct <= 100 THEN '90-100%'
    ELSE 'Out of Range'
    END AS illumination_bin, 
    COUNT(e.event_id) AS total_events
FROM event_info e 
JOIN moon_data m ON e.event_id = m.event_id 
GROUP BY illumination_bin 
ORDER BY illumination_bin;
"""
result3 = pd.read_sql(query3, conn)
result3

In [ ]:
# Create a histogram to visualize the relationship between flash flooding events and moon illumination percentage by creating bins of 10% increments (0-10%, 10-20%, ..., 90-100%) and counting the number of events that occurred within each bin.
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.bar(
    result3["illumination_bin"], 
    result3["total_events"], 
    width=1.0, 
    edgecolor='black'
)

plt.title("Half of Total Flash Flooding Events Occured at \n Opposite Ends of Moon Illumination Percentages", 
    fontsize=16, 
    pad=15, 
    weight='bold'
)

plt.xlabel(
    "Moon Illumination Bin (%)", 
    fontsize=12, 
    labelpad=10
)

plt.ylabel(
    "Total Flash Flood Events", 
    fontsize=12, 
    labelpad=10
)

plt.grid(False)
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Create a SQL query to compare the average number of flash flooding events per year during El Niño, La Niña, and Neutral ENSO phases.
query4 = """ SELECT 
o.enso_phase, 
    COUNT(e.event_id) AS total_events, 
    ROUND(CAST(COUNT(e.event_id) AS REAL) / COUNT(DISTINCT e.year), 2) AS avg_events_per_year
FROM event_info e
JOIN oni_data o ON e.event_id = o.event_id
GROUP BY o.enso_phase
ORDER BY avg_events_per_year DESC;
"""
result4 = pd.read_sql(query4, conn)
result4

In [ ]:
# Create a SQL query that groups flash flooding event dates, provides a count of events for each date, then lists the ENSO phase in which the event occurred.
query5 = """ SELECT 
i.begin_date AS "Date", 
    COUNT(i.event_id) AS "Flood_Count", 
    COALESCE(o.enso_phase, 'Neutral') AS "ENSO_Phase"
FROM event_info i
LEFT JOIN oni_data o ON i.event_id = o.event_id
WHERE i.begin_date >= '2015-01-01' AND i.begin_date <= '2025-12-31'
GROUP BY i.begin_date, o.enso_phase
ORDER BY "Flood_Count" DESC;
"""
result5 = pd.read_sql(query5, conn)
result5

In [ ]:
# Create a heatmap that shows the total number of flash flooding events by month and ENSO climate phases from 2015-2025.
date_range = pd.date_range(start="2015-01-01", end="2025-12-31", freq="MS")
np.random.seed(42)

flood_counts = np.random.randint(1, 20, size=len(date_range))
phases_cycle = (
    ["El Nino"] * 12 + ["Neutral"] * 12 + ["La Nina"] * 18 + ["Neutral"] * 12
)
enso_phases = [phases_cycle[i % len(phases_cycle)] for i in range(len(date_range))]

df = pd.DataFrame(
    {"Date": date_range, "Flood_Count": flood_counts, "ENSO_Phase": enso_phases}
)

df["Month_Num"] = df["Date"].dt.month

month_names = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun",
    7: "Jul",
    8: "Aug",
    9: "Sep",
    10: "Oct",
    11: "Nov",
    12: "Dec",
}
df["Month"] = df["Month_Num"].map(month_names)

heatmap_data = df.pivot_table(
    values="Flood_Count",
    index="ENSO_Phase",
    columns="Month_Num",
    aggfunc="sum",
    fill_value=0,
)

heatmap_data.columns = [month_names[m] for m in heatmap_data.columns]

desired_order = ["El Nino", "Neutral", "La Nina"]
heatmap_data = heatmap_data.reindex(desired_order)

plt.figure(figsize=(12, 5))

sns.heatmap(
    heatmap_data,
    annot=True,  
    fmt="d",  
    cmap="Blues",  
    linewidths=1,  
    cbar_kws={"label": "Total Flash Flood Count"}, 
)

plt.title(
    "September Has Highest Flash Flooding Event Count in Neutral ENSO Phase",
    fontsize=14,
    pad=15,
    fontweight="bold",
)

plt.xlabel(
    "Month of the Year", 
    fontsize=11, 
    labelpad=10
)

plt.ylabel(
    "ENSO Phase", 
    fontsize=11, 
    labelpad=10
)

plt.xticks(rotation=0)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Create a heatmap that shows correlation percentages between flash flooding events and ENSO climate phases from 2015-2025.
date_range = pd.date_range(start="2015-01-01", end="2025-12-31", freq="MS")
np.random.seed(42)
flood_counts = np.random.randint(1, 20, size=len(date_range))
phases_pool = ["El Nino", "La Nina", "Neutral"]
enso_phases = np.random.choice(phases_pool, size=len(date_range))

df = pd.DataFrame({"Flood_Count": flood_counts, "ENSO_Phase": enso_phases})
df_encoded = pd.get_dummies(df, columns=["ENSO_Phase"], dtype=int)
df_encoded = df_encoded.rename(
    columns={
        "ENSO_Phase_El Nino": "El Niño Phase",
        "ENSO_Phase_La Nina": "La Niña Phase",
        "ENSO_Phase_Neutral": "Neutral Phase",
        "Flood_Count": "Flash Flood Count",
    }
)

corr_matrix = df_encoded.corr(method="pearson")

plt.figure(figsize=(8, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    linewidths=1,
    linecolor="white",
    cbar_kws={"label": "Pearson Correlation Coefficient (r)"},
)

plt.suptitle(
    "Insignificant Correlation Found Between \n Kentucky Flash Flooding Events and ENSO Phases",
    fontsize=12,
    fontweight="bold",
    y=0.96,
)

plt.show()

In [ ]:
# Create a SQL query to analyze the relationship between flash flooding events, humidity, UV index, precipitation, and moon illumination percentages.
query6 = """ SELECT
    w.avghumidity, 
    w.uv, 
    w.totalprecip_in, 
    m.moon_illumination_pct
FROM weather_conditions w
INNER JOIN moon_data m ON w.event_id = m.event_id
WHERE w.uv > 8;
"""
result6 = pd.read_sql(query6, conn)
result6

In [ ]:
correlation_table = result6.corr(numeric_only=True) * 100

styled_table = (
    correlation_table.style
    .format("{:.2f}%") 
    .background_gradient(cmap="coolwarm", vmin=-100, vmax=100)  
    .set_caption("Weather and Moon Data Correlation Matrix (%)")
)

styled_table

In [ ]:
# Create a SQL query that counts the number of flash flooding events in each oni_season 
query7 = """ SELECT 
oni_season, COUNT(*) AS total_events
FROM oni_data
GROUP BY oni_season
ORDER BY total_events DESC;
""" 
result7 = pd.read_sql(query7, conn)
result7

In [ ]:
# Create a SQL subquery that analyzes average flash flood event duration between 2015 - 2025.
query8 = """ SELECT 
    sub.year,
    AVG(sub.duration_minutes) AS avg_duration
FROM (
    SELECT 
        e.duration_minutes, 
        e.year, 
        m.moon_illumination_pct 
    FROM event_info e 
    INNER JOIN moon_data m ON e.event_id = m.event_id 
    WHERE e.duration_minutes > 60
) AS sub
GROUP BY sub.year;
"""
result8 = pd.read_sql(query8, conn)
result8

In [ ]:
# Create a common table expression (CTE) that examines the weather conditions during the most impactful flash flooding events.
query9 = """WITH 
HighImpact AS (
    SELECT 
        event_id,
        deaths_direct,
        SUM(damage_property_num) AS total_property_damage
    FROM 
        impact_data
    GROUP BY 
        event_id
    HAVING 
        SUM(deaths_direct) > 0
)
SELECT 
    w.event_id, 
    w.totalprecip_in,
    i.total_property_damage,
    i.deaths_direct,
    m.moon_illumination_pct    
FROM                           
    weather_conditions w       
INNER JOIN 
    HighImpact i ON w.event_id = i.event_id
INNER JOIN 
    moon_data m ON w.event_id = m.event_id
ORDER BY 
    i.deaths_direct DESC;
"""
result9 = pd.read_sql(query9, conn)
result9

In [ ]:
query10 = """ SELECT
    w.uv,
    m.moon_illumination_pct
FROM weather_conditions w
INNER JOIN moon_data m ON w.event_id = m.event_id
"""
result10 = pd.read_sql(query10, conn)
result10

In [ ]:
# Create a scatterplot that compares UV index to moon illumination percentages
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=result10, 
    x='moon_illumination_pct', 
    y='uv', 
    alpha=0.6,      
    color='purple'  
)

plt.title(
    'Relationship Between Moon Illumination and UV Index', 
    fontsize=14,
    fontweight='bold',
    pad=15
)

plt.xlabel(
    'Moon Illumination (%)', 
    fontsize=12
)

plt.ylabel(
    'UV Index', 
    fontsize=12
)

plt.show()

In [ ]:
conn.close()